# Пример - Спектры и диагонализация в алгебре полиформ

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **ba4e62a**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/ba4e62a1c1bc7b1d7957fc33fdd3d6ae45b9cfcf.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Шесть точных примеров диагонализации. Восстанавливаем лапласиан, гриниан и полиформы по модам, превращаем путь в цикл преобразованием Чебышёва и проверяем, какие данные теряются при диагональной проекции.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra ba4e62a](https://github.com/magin67/polyform_algebra/commit/ba4e62a1c1bc7b1d7957fc33fdd3d6ae45b9cfcf).


## Диагонализация пути P3

Для пути $a-b-c$ находим проекторы двух ненулевых мод. Восстанавливаем лапласиан и гриниан и сравниваем первый и второй грейды экспонент, построенных по модам и по коэффициентам пар вершин.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F-%D0%B2-%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D0%B5-%D0%BF%D1%80%D0%B5%D0%B4%D1%81%D1%82%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C#%D0%B2%D1%81%D0%B5-%D0%B3%D1%80%D0%B5%D0%B9%D0%B4%D1%8B-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%B4%D0%BB%D1%8F-%D0%BF%D1%83%D1%82%D0%B8-p3).

In [3]:
from fractions import Fraction as Q

from sympy import Matrix, cos, pi, simplify, sqrt, zeros

from polyform_algebra.core import (
    InvalidInput, equivalent, exterior, form_exp, grade_part, green, laplacian,
    laplacian_form, network, network_from_laplacian, polar_form, resistance,
)
from polyform_algebra.duality import greenian_form
from polyform_algebra.spectral import (
    chebyshev_laplacian, fixed_basis_projection, laplacian_projectors,
)
from polyform_algebra.toponomes import laplacian_polynomial


def path(labels):
    return network(tuple(labels), [(a, b, 1) for a, b in zip(labels, labels[1:])])


def cycle(labels):
    return network(tuple(labels), [(a, b, 1) for a, b in zip(labels, labels[1:] + labels[:1])])


def lap_matrix(graph):
    return Matrix(laplacian(graph).value.values)


def projector(mode):
    vector = Matrix(mode)
    return vector * vector.T / vector.dot(vector)


def modal_form(context, mode):
    vector = exterior(context, tuple(((i,), value) for i, value in enumerate(mode) if value))
    return polar_form(vector, vector).value * Q(1, 2 * sum(value**2 for value in mode))


def modal_sum(context, modes, coefficients):
    first, *rest = [modal_form(context, mode) * coefficient
                    for mode, coefficient in zip(modes, coefficients, strict=True)]
    return sum(rest, first)


p3 = path("abc")
modes = ((1, 0, -1), (1, -2, 1))
p3_blocks = laplacian_projectors(p3)
assert set(p3_blocks) == {0, 1, 3}
assert p3_blocks[1] == projector(modes[0])
assert p3_blocks[3] == projector(modes[1])
assert p3_blocks[1] + 3*p3_blocks[3] == lap_matrix(p3)
assert p3_blocks[1] + p3_blocks[3]/3 == Matrix(green(p3).value.values)

for weights, direct in (((1, 3), laplacian_form(p3).value),
                        ((1, Q(1, 3)), greenian_form(p3).value)):
    from_modes = form_exp(modal_sum(p3.context, modes, weights)).value
    from_pairs = form_exp(direct).value
    for grade in (1, 2):
        left = grade_part(from_modes, grade).value
        right = grade_part(from_pairs, grade).value
        assert equivalent(left, right).value
print("P3: eigenvalues 0, 1, 3; Laplacian and Greenian grades 1 and 2 agree")

P3: eigenvalues 0, 1, 3; Laplacian and Greenian grades 1 and 2 agree


## Кратное собственное число цикла C4

Меняем базис в двумерном подпространстве для $\lambda=2$. Проверяем, что проектор, второй грейд и все попарные резистенсы совпадают при обоих выборах базиса.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F-%D0%B2-%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%BE%D0%B1%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%B4%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0,-%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D1%8B-%D0%B8-%D0%B2%D0%BE%D1%81%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC#%D0%B2%D1%82%D0%BE%D1%80%D0%BE%D0%B9-%D0%B3%D1%80%D0%B5%D0%B9%D0%B4-c4-%D0%BA%D0%B0%D0%BA-%D1%83%D1%87%D0%B8%D1%82%D1%8B%D0%B2%D0%B0%D0%B5%D1%82%D1%81%D1%8F-%D1%87%D0%B8%D1%81%D0%BB%D0%BE-%D0%BD%D0%B0%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B9).

In [4]:
c4 = cycle("abcd")
blocks = laplacian_projectors(c4)
assert {value: block.trace() for value, block in blocks.items()} == {0: 1, 2: 2, 4: 1}
original_modes = ((1, 0, -1, 0), (0, 1, 0, -1))
rotated_modes = ((1, 1, -1, -1), (1, -1, -1, 1))
mode4 = (1, -1, 1, -1)
for basis in (original_modes, rotated_modes):
    assert sum(map(projector, basis), zeros(4)) == blocks[2]
    modal_exponential = form_exp(modal_sum(c4.context, (*basis, mode4), (2, 2, 4))).value
    edge_exponential = form_exp(laplacian_form(c4).value).value
    assert equivalent(grade_part(modal_exponential, 2).value, grade_part(edge_exponential, 2).value).value
    reconstructed = sum(map(projector, basis), zeros(4))/2 + projector(mode4)/4
    by_mode = Matrix(4, 4, lambda i, j: reconstructed[i, i] + reconstructed[j, j] - 2*reconstructed[i, j])
    assert by_mode == Matrix(resistance(c4).value.values)
print("C4: rank-two projector, second grade and resistances survive a basis change")

C4: rank-two projector, second grade and resistances survive a basis change


## Собственные моды P6 и C6

Сверяем точные собственные числа и кратности, проверяем косинусные моды пути. Для противоположных вершин $a,d$ цикла суммируем вклады собственных подпространств в резистенс.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F-%D0%B2-%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%BE%D0%B1%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%BC%D0%BE%D0%B4%D1%8B-%D0%BF%D1%83%D1%82%D0%B5%D0%B9-%D0%B8-%D1%86%D0%B8%D0%BA%D0%BB%D0%BE%D0%B2#%D0%BA%D0%B0%D0%BA%D0%B8%D0%B5-%D0%BC%D0%BE%D0%B4%D1%8B-%D1%83%D1%87%D0%B0%D1%81%D1%82%D0%B2%D1%83%D1%8E%D1%82-%D0%B2-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%B5).

In [5]:
p6 = path("abcdef")
c6 = cycle("abcdef")
for graph, expected in (
    (p6, (0, 2-sqrt(3), 1, 2, 3, 2+sqrt(3))),
    (c6, (0, 1, 1, 3, 3, 4)),
):
    spectrum = laplacian_projectors(graph)
    for value in set(expected):
        assert spectrum[value].trace() == expected.count(value)
    assert len(spectrum) == len(set(expected))

for k in range(6):
    mode = Matrix([cos(pi*k*(i+Q(1, 2))/6) for i in range(6)])
    eigenvalue = 2 - 2*cos(pi*k/6)
    assert (lap_matrix(p6)*mode - eigenvalue*mode).applyfunc(simplify) == zeros(6, 1)

c6_blocks = laplacian_projectors(c6)
difference = Matrix([1, 0, 0, -1, 0, 0])
contributions = {value: (difference.T*block*difference)[0]/value
                 for value, block in c6_blocks.items() if value}
assert sum(contributions.values()) == resistance(c6, pairs=(("a", "d"),)).value.values[0] == Q(3, 2)
print("C6 opposite-pair resistance contributions:", {str(key): str(value) for key, value in contributions.items()})

C6 opposite-pair resistance contributions: {'1': '4/3', '3': '0', '4': '1/6'}


## Преобразование Чебышёва от P6 к C6

Вычисляем преобразование $4L-L^2$ и после перестановки вершин сравниваем всю матрицу с лапласианом $C_6$. Для третьего шага проверяем появление второй компоненты.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F-%D0%B2-%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%87%D0%B5%D0%B1%D1%8B%D1%88%D1%91%D0%B2%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0---%D0%BE%D1%82-%D0%BF%D1%83%D1%82%D0%B8-%D0%BA-%D1%86%D0%B8%D0%BA%D0%BB%D1%83#%D1%83%D0%B4%D0%B2%D0%BE%D0%B5%D0%BD%D0%B8%D0%B5-%D1%83%D0%B3%D0%BB%D0%B0-%D0%BF%D1%80%D0%B5%D0%B2%D1%80%D0%B0%D1%89%D0%B0%D0%B5%D1%82-%D0%BF%D1%83%D1%82%D1%8C-%D0%B2-%D1%86%D0%B8%D0%BA%D0%BB).

In [6]:
original = lap_matrix(p6)
doubled = chebyshev_laplacian(p6, 2)
assert doubled == 4*original - original**2
order = [p6.context.labels.index(label) for label in "abdfec"]
assert doubled.extract(order, order) == lap_matrix(c6)
assert doubled.charpoly().all_coeffs() == lap_matrix(c6).charpoly().all_coeffs()
assert len(network_from_laplacian(chebyshev_laplacian(p6, 3).tolist(), p6.context.labels).value.components) == 2
print("P6 doubled: cycle order a-b-d-f-e-c-a; the step-three transform disconnects")

P6 doubled: cycle order a-b-d-f-e-c-a; the step-three transform disconnects


## Конус в фиксированном базисе P3

По двум спектральным координатам $a,b$ восстанавливаем проводимости. Проверяем условие $a\geq0$ и $0\leq b\leq3a$ внутри конуса, на его границе и вне его.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F-%D0%B2-%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9-%D0%BA%D0%BE%D0%BD%D1%83%D1%81-%D1%84%D0%B8%D0%BA%D1%81%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D0%BE%D0%B3%D0%BE-%D0%B1%D0%B0%D0%B7%D0%B8%D1%81%D0%B0#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D1%81%D0%BE%D0%B1%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%8B%D0%B9-%D0%B1%D0%B0%D0%B7%D0%B8%D1%81-%D0%BF%D1%83%D1%82%D0%B8-p3).

In [7]:
for a, b in ((0, 0), (2, 0), (1, 3), (2, 3), (1, 4), (-1, 0)):
    candidate = a*p3_blocks[1] + b*p3_blocks[3]
    conductances = tuple(-candidate[i, j] for i, j in ((0, 1), (0, 2), (1, 2)))
    assert conductances == (Q(b, 3), Q(3*a-b, 6), Q(b, 3))
    inside = a >= 0 and 0 <= b <= 3*a
    assert inside == all(value >= 0 for value in conductances)
    if inside:
        rebuilt = network_from_laplacian(candidate.tolist(), p3.context.labels).value
        assert fixed_basis_projection(rebuilt, modes) == ((a, b), candidate)
    print("P3 cone", (a, b), "conductances:", tuple(map(str, conductances)))

P3 cone (0, 0) conductances: ('0', '0', '0')
P3 cone (2, 0) conductances: ('0', '1', '0')
P3 cone (1, 3) conductances: ('1', '0', '1')
P3 cone (2, 3) conductances: ('1', '1/2', '1')
P3 cone (1, 4) conductances: ('4/3', '-1/6', '4/3')
P3 cone (-1, 0) conductances: ('0', '-1/2', '0')


## Одинаковая проекция разных графов

У двух взвешенных графов совпадают все диагональные координаты в фиксированном базисе. Сравниваем исходные корневые полиномы и проверяем знак проводимости матрицы проекции.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F-%D0%B2-%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%86%D0%B8%D1%8F-%D0%BD%D0%B0-%D1%84%D0%B8%D0%BA%D1%81%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D1%8B%D0%B9-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9-%D0%B1%D0%B0%D0%B7%D0%B8%D1%81-%D0%B8-%D0%BF%D0%BE%D1%82%D0%B5%D1%80%D1%8F-%D0%B8%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%B8#%D1%80%D0%B0%D0%B7%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D1%8B-%D1%81-%D0%BE%D0%B4%D0%B8%D0%BD%D0%B0%D0%BA%D0%BE%D0%B2%D0%BE%D0%B9-%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%86%D0%B8%D0%B5%D0%B9).

In [8]:
rotated = ((2, 1, -2, -1), (-1, 2, 1, -2), (1, -1, 1, -1))


def weighted_cycle_family(r):
    return network(tuple("abcd"), [
        ("a", "b", -(2+3*r)/10), ("c", "d", -(2+3*r)/10),
        ("a", "d", (22+3*r)/10), ("b", "c", (22+3*r)/10),
        ("a", "c", (17-2*r)/5), ("b", "d", (8+2*r)/5),
    ])


first, second = (weighted_cycle_family(Q(r)) for r in (-4, -3))
values, projected = fixed_basis_projection(first, rotated)
assert fixed_basis_projection(second, rotated) == (values, projected)
assert values == (10, 4, 4)
assert lap_matrix(first) != lap_matrix(second)
assert laplacian_polynomial(first) == (1, 18, 80, 96)
assert laplacian_polynomial(second) == (1, 18, 87, 124)
assert -projected[0, 1] == -Q(1, 5)
try:
    network_from_laplacian(projected.tolist(), tuple("abcd"))
except InvalidInput:
    pass
else:
    raise AssertionError("the projected matrix has a negative conductance")
print("Same projection (10, 4, 4); different rooted polynomials:",
      tuple(map(str, laplacian_polynomial(first))), tuple(map(str, laplacian_polynomial(second))))

Same projection (10, 4, 4); different rooted polynomials: ('1', '18', '80', '96') ('1', '18', '87', '124')
